# Create Python 3.11 ENV

In [ ]:
!conda create -n femr-py311 python=3.11 -y
!conda run -n femr-py311 python -m pip install ipykernel
!conda run -n femr-py311 python -m pip install torch==2.1.2 \
    --index-url https://download.pytorch.org/whl/cu121
!conda run -n femr-py311 python -m ipykernel install \
    --user \
    --name femr-py311 \
    --display-name "Python 3.11 - FEMR"

!conda run -n femr-py311 python -m pip install femr==0.2.3 datasets==2.15.0 xformers transformers==4.35.2
!conda run -n femr-py311 python -m pip install meds_reader

In [ ]:
!/opt/conda/envs/femr-py311/bin/python3.11 -m pip install --no-cache-dir xformers 

In [ ]:
import sys

!/opt/conda/envs/femr-py311/bin/python3.11 -m pip uninstall -y xformers

!/opt/conda/envs/femr-py311/bin/python3.11 -m pip install \
    --no-cache-dir \
    --no-deps \
    --index-url https://download.pytorch.org/whl/cu121 \
    "xformers==0.0.23.post1"

In [ ]:
!meds_reader_convert /home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort /home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort_reader --num_threads 4

In [ ]:
!pip install MEDS-Inspect

# Imports

In [ ]:
import os
import re
import pandas as pd
import numpy as np

try:
    from google.cloud import bigquery
except ImportError:
    bigquery = None

import subprocess

cmd = """
source /home/jupyter/load-env.sh >/dev/null
env
"""

result = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

print("WORKSPACE_CDR:", os.environ.get("WORKSPACE_CDR"))
print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET"))

bucket = os.getenv("WORKSPACE_BUCKET")
cdr = os.environ.get("WORKSPACE_CDR")

if cdr is None:
    raise EnvironmentError(
        "WORKSPACE_CDR is not set. This script should be run inside an All of Us workspace."
    )

use_bqstorage = ("BIGQUERY_STORAGE_API_ENABLED" in os.environ)

# Load Model

In [ ]:
from huggingface_hub import notebook_login
import femr.models.transformer
import torch
import femr.models.tokenizer
import femr.models.processor
import datetime

notebook_login()

In [ ]:
model_name = "StanfordShahLab/motor-t-base"

# Load tokenizer / batch loader
# motor_tokenizer = femr.models.tokenizer.FEMRTokenizer.from_pretrained(model_name)
# motor_batch_processor = femr.models.processor.FEMRBatchProcessor(motor_tokenizer)

# Load model
motor_model = femr.models.transformer.FEMRModel.from_pretrained(model_name)

In [ ]:
import inspect
import femr.models.transformer as femr_transformer

print(inspect.signature(
    femr_transformer.FEMREncoderLayer.__init__
))

print(inspect.signature(
    femr_transformer.FEMREncoderLayer.forward
))

print(inspect.signature(
    femr_transformer.FEMRTransformer.forward
))

print(inspect.getsource(
    femr.models.transformer.FEMREncoderLayer
))

print(inspect.getsource(
    femr_transformer.FEMRTransformer
))

In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
model = AutoModelForMaskedLM.from_pretrained("boltuix/bert-mini")
tokenizer = AutoTokenizer.from_pretrained("boltuix/bert-mini")

## Full Model Pipeline

In [ ]:
import torch
import torch.nn as nn

import torch
import torch.nn as nn
import torch.nn.functional as F
import xformers

from torch.nn.utils.rnn import pad_sequence

import femr.models.transformer
from femr.models.transformer import fixed_pos_embedding

### Survey Cross attn

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence


class SurveyCrossAttn(nn.Module):
    def __init__(
        self,
        attn_heads,
        survey_dim,
        ehr_dim,
        dropout=0.1,
    ):
        super().__init__()

        if ehr_dim % attn_heads != 0:
            raise ValueError(
                f"ehr_dim={ehr_dim} must be divisible by "
                f"attn_heads={attn_heads}"
            )

        self.survey_embed_proj = (
            nn.Identity()
            if survey_dim == ehr_dim
            else nn.Linear(survey_dim, ehr_dim)
        )

        self.ehr_norm = femr.models.rmsnorm.RMSNorm(ehr_dim)
        self.survey_norm = femr.models.rmsnorm.RMSNorm(ehr_dim)
        self.output_norm = femr.models.rmsnorm.RMSNorm(ehr_dim)

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=ehr_dim,
            num_heads=attn_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)

        
    def forward(
        self,
        ehr_hidden,
        survey_hidden,
        patient_lengths,
        survey_attention_mask=None,
    ):
        """
        ehr_hidden:
            [sum(patient_lengths), ehr_dim]

        survey_hidden:
            [batch_size, survey_length, survey_dim]

        patient_lengths:
            [batch_size]

        survey_attention_mask:
            [batch_size, survey_length]
            1 = valid survey token
            0 = padding
        """

        # Keep a tensor on the same device as the EHR hidden states.
        patient_lengths_device = patient_lengths.to(
            device=ehr_hidden.device,
            dtype=torch.long,
        )

        # torch.split requires Python integers.
        patient_lengths_list = (
            patient_lengths_device.detach().cpu().tolist()
        )

        if sum(patient_lengths_list) != ehr_hidden.shape[0]:
            raise ValueError(
                "The sum of patient_lengths must equal the number "
                f"of packed EHR tokens. Got "
                f"{sum(patient_lengths_list)} and "
                f"{ehr_hidden.shape[0]}."
            )

        if survey_hidden.shape[0] != len(patient_lengths_list):
            raise ValueError(
                "The EHR and survey batch sizes do not match. "
                f"Got {len(patient_lengths_list)} EHR sequences "
                f"and {survey_hidden.shape[0]} survey sequences."
            )

        # Split packed EHR states into one sequence per patient.
        ehr_sequences = torch.split(
            ehr_hidden,
            patient_lengths_list,
            dim=0,
        )

        # [batch_size, max_ehr_length, ehr_dim]
        padded_ehr = pad_sequence(
            ehr_sequences,
            batch_first=True,
            padding_value=0.0,
        )

        max_ehr_length = padded_ehr.shape[1]

        # [batch_size, max_ehr_length]
        # True indicates a real EHR position.
        ehr_valid_mask = (
            torch.arange(
                max_ehr_length,
                device=ehr_hidden.device,
            ).unsqueeze(0)
            < patient_lengths_device.unsqueeze(1)
        )

        # [batch_size, survey_length, ehr_dim]
        survey_hidden = self.survey_embed_proj(survey_hidden)

        query = self.ehr_norm(padded_ehr)
        key_value = self.survey_norm(survey_hidden)

        survey_padding_mask = None

        if survey_attention_mask is not None:
            survey_attention_mask = survey_attention_mask.to(
                device=survey_hidden.device
            ).bool()

            if survey_attention_mask.shape != survey_hidden.shape[:2]:
                raise ValueError(
                    "survey_attention_mask must have shape "
                    "[batch_size, survey_length]."
                )

            # Every patient must have at least one survey token.
            if (~survey_attention_mask.any(dim=1)).any():
                raise ValueError(
                    "At least one patient has no valid survey tokens."
                )

            # MultiheadAttention:
            # True means ignore this key/value position.
            survey_padding_mask = ~survey_attention_mask

        # Batched cross-attention:
        #
        # Q = [B, max_EHR_length, ehr_dim]
        # K = [B, survey_length, ehr_dim]
        # V = [B, survey_length, ehr_dim]
        cross_output, _ = self.cross_attention(
            query=query,
            key=key_value,
            value=key_value,
            key_padding_mask=survey_padding_mask,
            need_weights=False,
        )

        # Residual connection around cross-attention.
        output = self.output_norm(
            padded_ehr + self.dropout(cross_output)
        )

        # Discard padded EHR query positions and restore FEMR's
        # packed ordering.
        packed_output = output[ehr_valid_mask]

        return packed_output

### Updated FEMR Encoder layer

In [ ]:
class FEMREncoderLayer(
    femr.models.transformer.FEMREncoderLayer
):
    def __init__(
        self,
        config,
        attn_heads,
        survey_dim,
        ehr_dim=None,
        dropout=0.1,
    ):
        super().__init__(config)

        if ehr_dim is None:
            ehr_dim = config.hidden_size

        if ehr_dim != config.hidden_size:
            raise ValueError(
                f"ehr_dim={ehr_dim} must match FEMR hidden size "
                f"{config.hidden_size}"
            )

        self.cross_attn = SurveyCrossAttn(
            attn_heads=attn_heads,
            survey_dim=survey_dim,
            ehr_dim=ehr_dim,
            dropout=dropout,
        )

    def forward(
        self,
        x,
        normed_ages,
        pos_embed,
        attn_bias,
        survey_hidden,
        patient_lengths,
        survey_attention_mask=None,
    ):
        # Original FEMR layer returns the update that is
        # normally added in FEMRTransformer.forward().
        x = super().forward(
            x=x,
            normed_ages=normed_ages,
            pos_embed=pos_embed,
            attn_bias=attn_bias,
        )

        # New survey cross-attention.
        
        # Q = EHR hidden states
        # K = survey hidden states
        # V = survey hidden states
        cross_attn = self.cross_attn(
            ehr_hidden=x,
            survey_hidden=survey_hidden,
            patient_lengths=patient_lengths,
            survey_attention_mask=survey_attention_mask,
        )

        return x + cross_attn # FEMR Residual connection

### Individual Gene prediction layer

In [ ]:
class GeneHeadLayer(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        num_selected_layers: int,
        num_genes: int,
        gene_embedding_dim: int = 32,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.num_genes = num_genes
        self.gene_embedding_dim = gene_embedding_dim

        concatenated_dim = hidden_size * num_selected_layers

        # Normalize each selected transformer representation separately.
        self.layer_norms = nn.ModuleList(
            [
                nn.LayerNorm(hidden_size)
                for _ in range(num_selected_layers)
            ]
        )

        self.dropout = nn.Dropout(dropout)

        # Produce one latent vector for every gene.
        #
        # [B, num_selected_layers * H]
        #                ->
        # [B, num_genes * gene_embedding_dim]
        self.to_gene_embeddings = nn.Linear(
            concatenated_dim,
            num_genes * gene_embedding_dim,
        )

        self.gene_embedding_norm = nn.LayerNorm(
            gene_embedding_dim
        )

        # One classifier vector per gene.
        #
        # gene_classifier_weights[g] is used only for gene g.
        self.gene_classifier_weights = nn.Parameter(
            torch.empty(num_genes, gene_embedding_dim)
        )

        self.gene_classifier_bias = nn.Parameter(
            torch.zeros(num_genes)
        )

        nn.init.xavier_uniform_(
            self.gene_classifier_weights
        )

    def forward(
        self,
        selected_patient_embeddings: list[torch.Tensor],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        selected_patient_embeddings:
            List containing one [B, H] tensor per selected layer.

        Returns:
            gene_logits:
                [B, num_genes]

            gene_embeddings:
                [B, num_genes, gene_embedding_dim]
        """

        if len(selected_patient_embeddings) != len(
            self.layer_norms
        ):
            raise ValueError(
                "Number of supplied layer outputs does not match "
                "num_selected_layers."
            )

        normalized_outputs = [
            norm(layer_output)
            for norm, layer_output in zip(
                self.layer_norms,
                selected_patient_embeddings,
            )
        ]

        # [B, H] + [B, H] + [B, H]
        #              ->
        # [B, 3H]
        combined = torch.cat(
            normalized_outputs,
            dim=-1,
        )

        combined = self.dropout(combined)

        # [B, 3H] -> [B, G * Dg]
        gene_embeddings = self.to_gene_embeddings(
            combined
        )

        # [B, G * Dg] -> [B, G, Dg]
        gene_embeddings = gene_embeddings.view(
            gene_embeddings.shape[0],
            self.num_genes,
            self.gene_embedding_dim,
        )

        gene_embeddings = self.gene_embedding_norm(
            gene_embeddings
        )

        # For each gene g:
        #
        # logit_g = gene_embedding_g · classifier_weight_g + bias_g
        #
        # [B, G, Dg] * [G, Dg]
        #                 ->
        # [B, G]
        gene_logits = (
            gene_embeddings
            * self.gene_classifier_weights.unsqueeze(0)
        ).sum(dim=-1)

        gene_logits = (
            gene_logits
            + self.gene_classifier_bias.unsqueeze(0)
        )

        return gene_logits, gene_embeddings

### Model Pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import xformers.ops


class GermLinePredModel(nn.Module):
    def __init__(
        self,
        survey_model,
        survey_tokenizer,
        ehr_model,
        ehr_tokenizer,
        xai_layers,
        num_selected_layers,
        num_genes,
        gene_embedding_dim: int = 32,
        survey_dim=None,
        ehr_dim=None,
        attn_heads=8,
        dropout=0.1,
        overall_loss_weight: float = 1.0,
        gene_loss_weight: float = 1.0,
    ):
        super().__init__()

        self.survey_embed_model = survey_model
        self.survey_embed_token = survey_tokenizer

        self.ehr_model = ehr_model
        self.ehr_token = ehr_tokenizer

        self.num_genes = num_genes
        self.overall_loss_weight = overall_loss_weight
        self.gene_loss_weight = gene_loss_weight

        ehr_transformer = self.ehr_model.transformer
        config = ehr_transformer.config

        num_transformer_layers = len(
            ehr_transformer.layers
        )

        # Convert negative indices, such as -1, to normal indices.
        normalized_xai_layers = []

        for layer_index in xai_layers:
            if layer_index < 0:
                layer_index = (
                    num_transformer_layers + layer_index
                )

            if not 0 <= layer_index < num_transformer_layers:
                raise ValueError(
                    f"Invalid XAI layer index {layer_index}. "
                    f"The transformer has "
                    f"{num_transformer_layers} layers."
                )

            normalized_xai_layers.append(layer_index)

        # Remove duplicates but preserve the supplied order.
        self.xai_layers = tuple(
            dict.fromkeys(normalized_xai_layers)
        )

        if len(self.xai_layers) != num_selected_layers:
            raise ValueError(
                f"num_selected_layers={num_selected_layers}, "
                f"but {len(self.xai_layers)} unique layer "
                f"indices were provided: {self.xai_layers}"
            )

        if survey_dim is None:
            survey_dim = (
                self.survey_embed_model.config.hidden_size
            )

        if ehr_dim is None:
            ehr_dim = config.hidden_size

        if ehr_dim != config.hidden_size:
            raise ValueError(
                f"ehr_dim={ehr_dim} must match CLMBR hidden "
                f"size {config.hidden_size}"
            )

        # Replace every original FEMR layer with the modified
        # FEMR + survey-cross-attention layer.
        modified_layers = nn.ModuleList()

        for original_layer in ehr_transformer.layers:
            modified_layer = FEMREncoderLayer(
                config=config,
                attn_heads=attn_heads,
                survey_dim=survey_dim,
                ehr_dim=ehr_dim,
                dropout=dropout,
            )

            loading_info = modified_layer.load_state_dict(
                original_layer.state_dict(),
                strict=False,
            )

            # Missing keys should correspond to newly added
            # cross-attention parameters.
            # Unexpected keys should normally be empty.
            if loading_info.unexpected_keys:
                raise RuntimeError(
                    "Unexpected FEMR checkpoint parameters: "
                    f"{loading_info.unexpected_keys}"
                )

            modified_layers.append(modified_layer)

        ehr_transformer.layers = modified_layers

        self.gene_pred_layer = GeneHeadLayer(
            hidden_size=ehr_dim,
            num_selected_layers=num_selected_layers,
            num_genes=num_genes,
            gene_embedding_dim=gene_embedding_dim,
        )

        # One logit for the overall germline-positive task.
        self.overall_classifier = nn.Sequential(
            nn.LayerNorm(ehr_dim),
            nn.Dropout(dropout),
            nn.Linear(ehr_dim, 1),
        )

    def forward(
        self,
        ehr_batch,
        survey_input_ids,
        survey_attention_mask,
        labels=None,
    ):
        # --------------------------------------------------
        # 1. Encode the survey sequences
        # --------------------------------------------------

        survey_outputs = self.survey_embed_model(
            input_ids=survey_input_ids,
            attention_mask=survey_attention_mask,
            return_dict=True,
        )

        survey_hidden = survey_outputs.last_hidden_state

        # survey_hidden:
        # [B, survey_length, survey_dim]

        # --------------------------------------------------
        # 2. Reproduce the FEMR transformer input path
        # --------------------------------------------------

        transformer = self.ehr_model.transformer
        config = transformer.config

        if not config.is_hierarchical:
            x = transformer.embed(
                ehr_batch["tokens"]
            )
        else:
            x = transformer.embed_bag(
                ehr_batch["hierarchical_tokens"],
                ehr_batch["token_indices"],
                ehr_batch["hierarchical_weights"],
            )

        # x:
        # [total_packed_events, ehr_dim]
        x = transformer.in_norm(x)

        patient_lengths = ehr_batch["patient_lengths"]

        if torch.any(patient_lengths <= 0):
            raise ValueError(
                "Every patient must contain at least one "
                "EHR sequence position."
            )

        if patient_lengths.sum().item() != x.shape[0]:
            raise ValueError(
                "The sum of patient_lengths does not match "
                "the packed EHR sequence length."
            )

        # Compute this only once.
        final_event_indices = (
            torch.cumsum(patient_lengths, dim=0) - 1
        ).long()

        normed_ages = ehr_batch["normalized_ages"]

        pos_embed = fixed_pos_embedding(
            ehr_batch["ages"],
            config.hidden_size // config.n_heads,
            x.dtype,
        )

        attn_bias = (
            xformers.ops.fmha.attn_bias.BlockDiagonalMask
            .from_seqlens(patient_lengths.tolist())
            .make_local_attention(
                config.attention_width
            )
        )

        # --------------------------------------------------
        # 3. EHR self-attention and survey cross-attention
        # --------------------------------------------------

        selected_embeddings_by_layer = {}

        final_layer_index = len(transformer.layers) - 1

        for layer_index, layer in enumerate(
            transformer.layers
        ):
            x = layer(
                x=x,
                normed_ages=normed_ages,
                pos_embed=pos_embed,
                attn_bias=attn_bias,
                survey_hidden=survey_hidden,
                patient_lengths=patient_lengths,
                survey_attention_mask=(
                    survey_attention_mask
                ),
            )

            # Save intermediate layers here.
            #
            # The final layer is saved after out_norm below.
            if (
                layer_index in self.xai_layers
                and layer_index != final_layer_index
            ):
                selected_embeddings_by_layer[
                    layer_index
                ] = x[final_event_indices]

        # Apply CLMBR's final normalization.
        x = transformer.out_norm(x)

        # Main patient representation:
        # [B, ehr_dim]
        patient_embeddings = x[final_event_indices]

        # If the final layer is selected, use its normalized
        # patient representation.
        if final_layer_index in self.xai_layers:
            selected_embeddings_by_layer[
                final_layer_index
            ] = patient_embeddings

        # Restore exactly the order specified in self.xai_layers.
        xai_embeddings = [
            selected_embeddings_by_layer[layer_index]
            for layer_index in self.xai_layers
        ]

        # --------------------------------------------------
        # 4. Overall and gene-specific predictions
        # --------------------------------------------------

        overall_logits = self.overall_classifier(
            patient_embeddings
        )

        # overall_logits:
        # [B, 1]

        gene_logits, gene_embeddings = (
            self.gene_pred_layer(xai_embeddings)
        )

        # gene_logits:
        # [B, num_genes]
        #
        # gene_embeddings:
        # [B, num_genes, gene_embedding_dim]

        if gene_logits.shape[-1] != self.num_genes:
            raise RuntimeError(
                f"Gene head returned {gene_logits.shape[-1]} "
                f"outputs, expected {self.num_genes}."
            )

        # First element = overall prediction.
        # Remaining elements = individual genes.
        joint_logits = torch.cat(
            [overall_logits, gene_logits],
            dim=-1,
        )

        # joint_logits:
        # [B, 1 + num_genes]

        # --------------------------------------------------
        # 5. Multi-task, multi-label loss
        # --------------------------------------------------

        loss = None
        overall_loss = None
        gene_loss = None

        if labels is not None:
            expected_shape = (
                joint_logits.shape[0],
                1 + self.num_genes,
            )

            if tuple(labels.shape) != expected_shape:
                raise ValueError(
                    f"labels must have shape {expected_shape}, "
                    f"but received {tuple(labels.shape)}."
                )

            labels = labels.float()

            overall_targets = labels[:, 0]
            gene_targets = labels[:, 1:]

            overall_loss = (
                F.binary_cross_entropy_with_logits(
                    overall_logits.squeeze(-1),
                    overall_targets,
                )
            )

            gene_loss = (
                F.binary_cross_entropy_with_logits(
                    gene_logits,
                    gene_targets,
                )
            )

            loss = (
                self.overall_loss_weight * overall_loss
                + self.gene_loss_weight * gene_loss
            )

        return {
            "loss": loss,
            "overall_loss": overall_loss,
            "gene_loss": gene_loss,
            "logits": joint_logits,
            "overall_logits": overall_logits,
            "gene_logits": gene_logits,
            "gene_embeddings": gene_embeddings,
            "patient_embeddings": patient_embeddings,
        }

# Load Data

## FEMR Dataloader

In [ ]:
import torch
from torch.utils.data import Dataset


class FullPatientDataset(Dataset):
    def __init__(
        self,
        subject_ids,
        database,
        ehr_processor,
        survey_by_subject,
        label_by_subject,
        survey_tokenizer,
        survey_max_length,
    ):
        self.subject_ids = list(subject_ids)
        self.database = database
        self.ehr_processor = ehr_processor
        self.survey_by_subject = survey_by_subject
        self.label_by_subject = label_by_subject
        self.survey_tokenizer = survey_tokenizer
        self.survey_max_length = survey_max_length

    def __len__(self):
        return len(self.subject_ids)

    def __getitem__(self, index):
        subject_id = self.subject_ids[index]
        subject = self.database[subject_id]
        subject_key = str(subject_id)

        # Complete patient EHR: no FEMR truncation or subsequence slicing.
        ehr = self.ehr_processor.convert_subject(
            subject,
            max_length=None,
            tensor_type="pt",
        )

        survey = self.survey_tokenizer(
            self.survey_by_subject[subject_key],
            truncation=True,
            max_length=self.survey_max_length,
            return_tensors="pt",
        )

        labels = torch.tensor(
            self.label_by_subject[subject_key],
            dtype=torch.float32,
        )

        if labels.shape != (74,):
            raise ValueError(
                f"Expected label shape (74,), got {labels.shape} "
                f"for subject {subject_id}"
            )

        return {
            "ehr": ehr,
            "survey_input_ids": survey["input_ids"].squeeze(0),
            "survey_attention_mask": survey[
                "attention_mask"
            ].squeeze(0),
            "labels": labels,
        }

## FEMR Collator

In [ ]:
class FullPatientCollator:
    def __init__(self, ehr_processor, survey_pad_token_id):
        self.ehr_processor = ehr_processor
        self.survey_pad_token_id = survey_pad_token_id

    def __call__(self, items):
        if len(items) != 1:
            raise ValueError(
                "This pipeline requires exactly one patient "
                "per forward pass."
            )

        item = items[0]

        # Adds FEMR's expected outer batch dimension.
        ehr_batch = self.ehr_processor.collate(
            [item["ehr"]]
        )["batch"]

        return {
            "ehr_batch": ehr_batch,
            "survey_input_ids": (
                item["survey_input_ids"].unsqueeze(0)
            ),
            "survey_attention_mask": (
                item["survey_attention_mask"].unsqueeze(0)
            ),
            "labels": item["labels"].unsqueeze(0),
        }

## Set up MEDS data for training

In [ ]:
import meds_reader
import inspect
import femr.splits
inspect.getsource(meds_reader.SubjectDatabase)

database = meds_reader.SubjectDatabase(
    "/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort_reader"
)


main_split = femr.splits.generate_hash_split(list(database), 97, frac_test=0.3)

train_split = femr.splits.generate_hash_split(main_split.train_patient_ids, 87, frac_test=0.3)

main_database = database.filter(main_split.train_patient_ids)
train_database = main_database.filter(train_split.train_patient_ids)
val_database = main_database.filter(train_split.test_patient_ids)



In [ ]:
import inspect
import importlib.metadata
import femr
import meds_reader

print("femr:", importlib.metadata.version("femr"))
print("meds_reader:", importlib.metadata.version("meds_reader"))

print(inspect.signature(
    femr.models.processor.FEMRBatchProcessor.convert_patient
))

print(femr.__file__)
print(meds_reader.__file__)

In [ ]:
genetic_df = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/entries_table_full_v8.csv')

In [ ]:
genetic_filt = genetic_df[['s', 'annotations.gene_symbol']]
genetic_filt.head()

In [ ]:
genes = genetic_filt['annotations.gene_symbol'].unique()

In [ ]:
genes

In [ ]:
genetic_filt.groupby(by='s').agg(lambda x: set([*x]))
genetic_filt["label"] = (
    genetic_filt["annotations.gene_symbol"]
    .astype(object)
    .map(lambda x: np.asarray(x == genes).astype(int).ravel())
)

In [ ]:
genetic_filt

## Finalize data loading

In [ ]:
from pathlib import Path

dir_path = Path('/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort/data/')
survey_df = pd.read_parquet('/home/jupyter/workspace/data_bucket/survey_data/survey.parquet')

In [ ]:
meds_patients = set()

for file in dir_path.glob("*/*.parquet"):
    meds = pd.read_parquet(file)
    patients = meds.subject_id.unique()
    meds_patients.update(patients)
shared_patients = set(survey_df["person_id"]) & meds_patients

In [ ]:
len(shared_patients)

In [ ]:
entrie_v8 = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/entries_table_full_v8.csv')

In [ ]:
entrie_v8.s.nunique()

In [ ]:
for file in dir_path.glob("*/*.parquet"):
    meds = pd.read_parquet(file)
    meds = meds[meds['subject_id'].isin(shared_patients)]
    meds.to_parquet(file, index=False)

survey_df = survey_df[survey_df['person_id'].isin(shared_patients)]
survey_df.to_parquet('/home/jupyter/workspace/data_bucket/survey_data/survey_filt.parquet')
genetic_df = genetic_df[genetic_df['s'].isin(shared_patients)]
genetic_df.to_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_filt_v9.csv')